In [9]:
# Single Jupyter cell. No tqdm. No ipywidgets. Symmetric seamless tiles with SD v1-5.

# Optional installs if needed:
# !pip -q install "diffusers>=0.30" "transformers>=4.44" accelerate safetensors torch torchvision

import os, sys, types

# 0) Quiet and non interactive env
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TQDM_NOTEBOOK"] = "0"

# 1) Stub tqdm everywhere to avoid any widget paths
class _NoTqdm:
    def __init__(self, iterable=None, *a, **k):
        self.iterable = iterable if iterable is not None else range(0)
    def __iter__(self): return iter(self.iterable)
    def update(self, *a, **k): pass
    def close(self): pass
    def set_description(self, *a, **k): pass
    def set_postfix(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass

def _install_tqdm_stub(name):
    m = types.ModuleType(name)
    m.tqdm = _NoTqdm
    m.trange = lambda n=0, *a, **k: range(n)
    sys.modules[name] = m

_install_tqdm_stub("tqdm")
_install_tqdm_stub("tqdm.std")
_install_tqdm_stub("tqdm.auto")
_install_tqdm_stub("tqdm.notebook")

# 2) Stub ipywidgets and its submodules so nothing tries to build a widget
w = types.ModuleType("ipywidgets")
widgets = types.ModuleType("ipywidgets.widgets")

class _DummyWidget:
    def __init__(self, *a, **k): pass
    def close(self): pass

# Common widget names used by tqdm.notebook
for mod in (w, widgets):
    mod.Widget = _DummyWidget
    mod.HBox = _DummyWidget
    mod.VBox = _DummyWidget
    mod.IntProgress = _DummyWidget
    mod.FloatProgress = _DummyWidget
    mod.HTML = _DummyWidget
    mod.Layout = _DummyWidget

w.widgets = widgets
sys.modules["ipywidgets"] = w
sys.modules["ipywidgets.widgets"] = widgets

# 3) Now import the libs
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

# 4) Device and dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device in ("cuda", "mps") else torch.float32

# 5) Load Stable Diffusion v1.5
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, safety_checker=None)
pipe.to(device)
try:
    pipe.set_progress_bar_config(disable=True)
except Exception:
    pass

# 6) Circular boundary conditions for seamless tiling
def set_circular_padding(module):
    for m in module.modules():
        if isinstance(m, torch.nn.Conv2d):
            m.padding_mode = "circular"

set_circular_padding(pipe.unet)
if hasattr(pipe, "vae"):
    if hasattr(pipe.vae, "encoder"):
        set_circular_padding(pipe.vae.encoder)
    if hasattr(pipe.vae, "decoder"):
        set_circular_padding(pipe.vae.decoder)

# 7) Enforce vertical and horizontal symmetry on latents at each step
def symmetry_callback(pipe, step_index, timestep, callback_kwargs):
    latents = callback_kwargs["latents"]
    latents = (
        latents
        + torch.flip(latents, dims=[-1])              # horizontal
        + torch.flip(latents, dims=[-2])              # vertical
        + torch.flip(torch.flip(latents, [-1]), [-2]) # both
    ) / 4.0
    callback_kwargs["latents"] = latents
    return callback_kwargs

# 8) Prompt tuned for repeating features
prompt = (
    "seamless geometric tessellation, interlocking arabesque motifs, repeating ornamental wallpaper, "
    "high detail, crisp edges, balanced colors, symmetry, tiling pattern"
)

# 9) Generation settings
width = 768
height = 768
steps = 30
guidance = 7.0
seed = 42
generator = torch.Generator(device=device).manual_seed(seed)

# 10) Generate one seamless symmetric tile
result = pipe(
    prompt=prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    width=width,
    height=height,
    generator=generator,
    callback_on_step_end=symmetry_callback,
    callback_on_step_end_tensor_inputs=["latents"],
)
tile = result.images[0]

# 11) Build a 3x3 tiling preview
def make_grid(img, n=3):
    w, h = img.size
    grid = Image.new("RGB", (w * n, h * n))
    for i in range(n):
        for j in range(n):
            grid.paste(img, (i * w, j * h))
    return grid

grid = make_grid(tile, n=3)

display(tile)
display(grid)

tile.save("sd_tile.png")
grid.save("sd_tile_3x3.png")
print("Saved sd_tile.png and sd_tile_3x3.png")


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

LookupError: <ContextVar name='shell_parent' at 0x10c384180>